In [19]:
import os
from io import StringIO
import dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from langchain_openai import AzureChatOpenAI

dotenv.load_dotenv()

True

In [ ]:
model: AzureChatOpenAI = None
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are an expert in ontology engineering an RDF onthology in Turtle format as ttl file.
You are helping a scientist map out large sets of information about material tests which can be (but not only):
- objects from a specific scientic domain with different classes
- properties with metrics or data ranges from that domain
- activities which can be performed on those objects
- capabilities that these objects can have
- relationships between objects from that domain

You should generate your output in the form of an onthology and only use the following RDF objects: owl:equivalentClass, rdfs:isDefinedBy, wl:ObjectProperty ,rdfs:comment, owl:Class, rdfs:subClassOf, rdfs:member, rdfs:Datatype, rdfs:label, rdfs:domain, owl:DatatypeProperty, rdf:Property, rdfs:range, rdfs:Literal
Approach the problem like this:
1. To start you should summarize the input to understand the context and the information that has been provided to make sure that this is reflected in the later onthology. 
2. Iterate on the number of terms until you end up with about 500 terms.
3. Generate a list of all types which are relevant for the onthology and should be reflected as a owl:Class
4. Add a label with a short description of not more than 200 characters and add this as rdfs:comment, rdfs:label to each class
5. Carefully examine the domain and overall relevant properties. Make sure you extract those properties and add them with owl:DatatypeProperty, rdfs:member, owl:ObjectProperty, rdfs:Literal, rdf:Property, rdfs:domain, rdfs:range to each class. Introduce custom properties where they make sense but try to limit the total amout of custom properties.
6. Make sure to link the owl:equivalentClass notion to comparable object types that share the same properties.
7. Identify specialisations of the collected types in the sense of a subclass and establish the relationship in rdfs:subClassOf

Some general hints:
- Make sure to also extract synonym and extract a boolean property if the indicated terms are deprecated.
- Then you should create a hierarchy of classes and properties according to common characteristics and properties for example for elements that are about composits, cold or cooling you should create a common class and then create subclasses for each of the terms.
- Add the numeric value to the label of the class or property. Do not use any special characters in the label.
- Make sure to cary over the definitions, numerations and value ranges and add them to the ontology. You should avoid creating single top level elements in the hierary with no children. Make sure to create common classes and properties for the terms that are related to each other.
- Your should ensure specialisation of terms dimensions of measurement. For example everything that is about time, energy, viscosity should be a specicalisation a common parent time, energy and viscosity.
- If there are common postfixes that refer not to the concept but rather to a property like for example *Deprecated in the sense of that this attribute is deprecated this should be reflected rather as an attribute and not part of the name.
- You should also look for common prefix or infix as meaning of grouping - for example everything that starts with Polymer.
- You should make sure to reflect all the meanings of the context of "energy" which can refer to surface energy, Gibbs free energy, activation energy, thermal energy applied in applications, and more. 
- You should make sure to reflect all the meanings in the context of "viscosity" could mean initial viscosity, dynamic viscosity, or other types depending on the situation.
- You should make sure to reflect all the meanings in the context of "time." It might refer to hold time, hold time, curing time, shelf life, duration within a testing process, or other time-related concepts.
- Do not add comments and do not use the uptick comment in the inline text to not break the xml
This is the content you should use as input:
"""

onthology_prompt = """Here is the existing onthology. Please extend and restructure it according to the new input so that there more hierarchy levels and the new terms are added to the correct classes and properties. Please make sure to also add the synonyms, deprecated boolean property and the definitions to the ontology. Provide the output in OWL (XML) format and only output the ontology and nothing else while minimizing characters - do not add line breaks or comments."""

In [37]:
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")

model = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        azure_deployment=os.getenv("AZURE_OPENAI_COMPLETION_DEPLOYMENT_NAME"),
        openai_api_version=os.getenv("AZURE_OPENAI_VERSION"),
        temperature=0
    )

reasoning_model = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        azure_deployment='o3-mini',
        openai_api_version=os.getenv("AZURE_OPENAI_VERSION"),
        temperature=1
    )

In [39]:
image_path = "../../data/iso_472_large.md"
onthology_file_path="onthology_iso_427_large.xml"
onthology_file_temp_path="onthology_iso_427_large_temp.ttl"
onthology_file_output_path="onthology_iso_427_large_output.ttl"

with open(image_path, "r") as file:
    markdown = file.read()

with open(onthology_file_path, "r") as file:
    onthology = file.read()

message = HumanMessage(
    content=[
        {"type": "text", "text": system_prompt + markdown + onthology_prompt + onthology},
    ],
)

response = reasoning_model.invoke([message])
print(response)

with open(onthology_file_temp_path, "w") as file:
    new_ontology = response.content.replace("```xml", "").replace("```", "")
    file.write(new_ontology)

fixPrompt = HumanMessage(
    content=[
        {"type": "text", "text": "your task it to validate the ontholoty in the owl format and fix all errors to make it compatible with Turtle files. Apply all changes, do not remove content and only output the fixed xml and nothing else, no summary: " + response.content },
    ],
)


response = reasoning_model.invoke([fixPrompt])
print(response)

with open(onthology_file_output_path, "w") as file:
    new_ontology = response.content.replace("```xml", "").replace("```", "")
    file.write(new_ontology)

content='<?xml version="1.0"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#" xmlns:xsd="http://www.w3.org/2001/XMLSchema#" xmlns:mat="http://example.org/materials#" xml:base="http://example.org/materials"><owl:Ontology rdf:about="http://example.org/materials"/>\n<!-- Core Domain Extensions -->\n<owl:Class rdf:about="#Macromolecule"><rdfs:label>Macromolecule</rdfs:label><rdfs:comment>Large molecule such as a polymer</rdfs:comment></owl:Class>\n<owl:Class rdf:about="#Resin"><rdfs:label>Resin</rdfs:label><rdfs:comment>Organic solid or semi-solid material, base for plastics</rdfs:comment><rdfs:subClassOf rdf:resource="#Material"/></owl:Class>\n<!-- Polymers by Chemistry -->\n<owl:Class rdf:about="#Polyolefin"><rdfs:label>Polyolefin</rdfs:label><rdfs:comment>Plastic derived from alkene monomers (ethylene, propylene)</rdfs:comment><rdfs:subClassOf rdf:resource="#Plastic"/></owl:Cla